# Bài 4 — Đo chất lượng RAG ⭐⭐

**Tuần 21** · Bài tập kèm theo: `ex03_eval_rag.py` · Chuẩn bị cho **P6**

Đây là bài quan trọng nhất Phase 08.

Ba bài trước xây được một hệ RAG chạy được. Bài này trả lời câu hỏi mà **mọi**
buổi phỏng vấn AI Engineer đều hỏi:

> Bạn vừa đổi chunk size và cách tìm kiếm. **Hệ thống tốt lên hay bạn vừa làm
> hỏng những câu đang chạy đúng?**

Không có bộ eval thì bạn chỉ có thể đoán. Và bạn sẽ đoán sai.

> 💡 Chạy được **không cần API key** — phần sinh câu trả lời dùng một trình giả
> lập, và mục 2 giải thích vì sao điều đó lại *tốt* cho việc đo lường.

In [ ]:
import sys
from pathlib import Path

import numpy as np

# Tim thu muc goc du an du notebook duoc mo tu dau
GOC = Path.cwd()
while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
    GOC = GOC.parent
P08 = GOC / "curriculum" / "08-rag"
sys.path.insert(0, str(P08))

from tien_ich import bat_utf8, nap_tai_lieu, nap_cau_hoi_vang
from nhung_gia_lap import nhung, nhung_nhieu, giai_thich, SO_CHIEU, TEN_CHIEU

bat_utf8()   # Windows: console cp1252 se lam sap moi lenh print tieng Viet

DOCS = nap_tai_lieu()
VANG = nap_cau_hoi_vang()

print(f"{len(DOCS)} tài liệu · {sum(len(v) for v in DOCS.values()):,} ký tự")
print(f"{len(VANG)} câu hỏi vàng · embedding giả lập {SO_CHIEU} chiều")

In [ ]:
def cat_co_dinh(van_ban, kich_thuoc, chong_lap=0):
    """Giong ex01_chunking.cat_co_dinh - dat o day de notebook chay doc lap."""
    if chong_lap >= kich_thuoc:
        raise ValueError("chong_lap phai nho hon kich_thuoc")
    buoc = kich_thuoc - chong_lap
    return [van_ban[i:i + kich_thuoc] for i in range(0, len(van_ban), buoc)]


def cat_kho(docs, kich_thuoc=600, chong_lap=120):
    """Cat toan bo kho -> (danh sach chunk, danh sach ten file tuong ung)."""
    chunks, nguon = [], []
    for ten, noi_dung in docs.items():
        for c in cat_co_dinh(noi_dung, kich_thuoc, chong_lap):
            chunks.append(c)
            nguon.append(ten)
    return chunks, nguon


def gop(s):
    """Gop moi khoang trang/xuong dong thanh mot dau cach."""
    return " ".join(s.split())


def co_trich_dan(cac_chunk, trich_dan):
    """Doan trich dan co nam trong cac chunk lay ve khong?"""
    kho = gop(" || ".join(cac_chunk))
    return gop(trich_dan) in kho

In [ ]:
import math
import re
from collections import Counter

MAU_TU = re.compile(r"[0-9a-zà-ỹđ]+(?:[-.][0-9a-zà-ỹđ]+)*")
tach_tu = lambda s: MAU_TU.findall(s.lower())


def dung_he_tim(kich_thuoc, chong_lap, cach):
    """Trả về (ham_tim, chunks, nguon) cho MỘT cấu hình RAG.

    `cach` = "vector" | "bm25" | "hybrid"
    """
    chunks, nguon = cat_kho(DOCS, kich_thuoc, chong_lap)
    M = nhung_nhieu(chunks)

    tu = [tach_tu(c) for c in chunks]
    tf = [Counter(t) for t in tu]
    dai = [len(t) for t in tu]
    dai_tb = sum(dai) / len(dai)
    df = Counter()
    for t in tu:
        df.update(set(t))

    def diem_bm25(q, k1=1.5, b=0.75):
        qt = tach_tu(q)
        out = []
        for i in range(len(chunks)):
            chuan = 1 - b + b * dai[i] / dai_tb
            s = 0.0
            for w in qt:
                n = tf[i].get(w, 0)
                if n:
                    idf = math.log(1 + (len(chunks) - df[w] + 0.5) / (df[w] + 0.5))
                    s += idf * n * (k1 + 1) / (n + k1 * chuan)
            out.append(s)
        return np.array(out)

    def tim(cau, k=3, k0=60):
        if cach == "vector":
            return list(np.argsort(-(M @ nhung(cau)))[:k])
        if cach == "bm25":
            return list(np.argsort(-diem_bm25(cau))[:k])
        diem = {}
        for bang in (np.argsort(-diem_bm25(cau)), np.argsort(-(M @ nhung(cau)))):
            for hang, i in enumerate(bang, 1):
                diem[int(i)] = diem.get(int(i), 0.0) + 1.0 / (k0 + hang)
        return sorted(diem, key=lambda i: (-diem[i], i))[:k]

    return tim, chunks, nguon


print("Sẵn sàng dựng nhiều cấu hình để so sánh.")

## 1. Bộ chấm điểm

Một hệ RAG hỏng ở **hai chỗ độc lập**. Nếu chỉ có một con số đúng/sai, bạn thấy
điểm tụt mà không biết đi sửa chỗ nào — và người mới gần như luôn đoán nhầm là
do prompt.

```
      retrieval sai                     sinh câu trả lời sai
      (lấy nhầm đoạn)                   (có đoạn đúng mà vẫn trả lời sai)
            │                                     │
      recall ------------------------- từ khoá + trích dẫn + từ chối
```

Nên điểm được tách làm ba phần:

| Phần | Trọng số | Đo bước nào |
|---|---|---|
| `recall == 1.0` | 0.4 | Retrieval lấy đủ đoạn cần |
| Đủ từ khoá bắt buộc | 0.4 | Câu trả lời nói đúng ý |
| Trích dẫn hợp lệ | 0.2 | Có dẫn nguồn, và nguồn có thật |

Riêng câu **không có đáp án**: 1.0 nếu từ chối, 0.0 nếu trả lời — không có
điểm thành phần, vì chỉ có một hành vi đúng duy nhất.

In [ ]:
CUM_TU_CHOI = ["không tìm thấy", "không có thông tin", "không đề cập",
               "tài liệu không", "không nói về"]
MAU_TRICH_DAN = re.compile(r"\[([1-9][0-9]*)\]")


def cham_mot_cau(cau_tra_loi, cac_chunk_lay, cau_vang):
    td = cau_vang["trich_dan"]
    kho = gop(" || ".join(cac_chunk_lay))
    recall = (sum(1 for t in td if gop(t) in kho) / len(td)) if td else 1.0

    tl = gop(cau_tra_loi).lower()
    du_tu_khoa = all(gop(t).lower() in tl for t in cau_vang["tu_khoa"])

    so = {int(x) for x in MAU_TRICH_DAN.findall(cau_tra_loi)}
    trich_dan_ok = bool(so) and not {n for n in so if n > len(cac_chunk_lay)}
    tu_choi = any(c in tl for c in CUM_TU_CHOI)

    if not td:
        diem = 1.0 if tu_choi else 0.0
    else:
        diem = 0.4 * (recall == 1.0) + 0.4 * du_tu_khoa + 0.2 * trich_dan_ok

    return {"recall": recall, "du_tu_khoa": du_tu_khoa,
            "trich_dan_ok": trich_dan_ok, "tu_choi": tu_choi,
            "diem": round(diem, 6)}


def chay_eval(ham_tra_loi):
    """ham_tra_loi(cau_hoi) -> (cau_tra_loi, cac_chunk_lay). Lỗi -> 0 điểm."""
    kq = []
    for c in VANG:
        ban_ghi = {"id": c["id"], "loai": c["loai"], "diem": 0.0,
                   "chi_tiet": {}, "loi": None}
        try:
            tl, lay = ham_tra_loi(c["cau_hoi"])
            ct = cham_mot_cau(tl, lay, c)
            ban_ghi["chi_tiet"], ban_ghi["diem"] = ct, ct["diem"]
        except Exception as e:
            ban_ghi["loi"] = f"{type(e).__name__}: {e}"
        kq.append(ban_ghi)
    return kq


def bao_cao(kq, ten=""):
    theo_loai = {}
    for r in kq:
        theo_loai.setdefault(r["loai"], []).append(r["diem"])
    tong = sum(r["diem"] for r in kq) / len(kq)
    dong = "  ".join(f"{l}={sum(v)/len(v):.2f}" for l, v in sorted(theo_loai.items()))
    print(f"{ten:<24} TỔNG={tong:.2f}   {dong}")
    return tong


print("Bộ chấm sẵn sàng.")

## 2. Trình sinh giả lập — và vì sao nó lại hữu ích

Để chạy được không cần API key, ta dùng một trình sinh **giả lập**: nó đọc các
đoạn lấy về, và nếu tìm thấy đoạn trích đúng thì viết câu trả lời kèm trích dẫn;
không thấy thì từ chối.

Nghe như gian lận. Nhưng nó có một tính chất rất giá trị:

> **Nó mô phỏng một trình sinh HOÀN HẢO.** Nên mọi điểm bị mất đều là lỗi
> retrieval, không lẫn lỗi sinh.

Đó chính là điều bạn muốn khi đang tinh chỉnh chunking và tìm kiếm: **cô lập
một biến**. Mục 7 sẽ cho model thật vào để thấy phần còn lại.

In [ ]:
def tao_trinh_sinh(ham_tim, chunks, nguon, k=3):
    """Trả về ham_tra_loi(cau_hoi) dùng cho chay_eval."""

    def tra_loi(cau_hoi):
        idx = ham_tim(cau_hoi, k)
        lay = [chunks[i] for i in idx]

        cau_vang = next(c for c in VANG if c["cau_hoi"] == cau_hoi)
        kho = gop(" || ".join(lay))
        thay = [t for t in cau_vang["trich_dan"] if gop(t) in kho]

        if not cau_vang["trich_dan"] or not thay:
            return ("Không tìm thấy thông tin trong tài liệu.", lay)

        # Trình sinh "hoàn hảo": nói đúng mọi từ khoá mà đoạn lấy về chống lưng
        tu_khoa = [t for t in cau_vang["tu_khoa"] if gop(t).lower() in kho.lower()]
        return (f'{" ".join(tu_khoa)} — theo tài liệu [1].', lay)

    return tra_loi


tim_A, chunk_A, nguon_A = dung_he_tim(300, 0, "vector")
kq_A = chay_eval(tao_trinh_sinh(tim_A, chunk_A, nguon_A))
bao_cao(kq_A, "A: 300/0, vector")

## 3. Baseline — con số 0.77 kia có nghĩa gì?

Trước khi ăn mừng hay thất vọng vì bất kỳ con số nào, phải có mốc so.

In [ ]:
def baseline_lay_dau(cau_hoi):
    """Không tìm kiếm gì cả: cứ lấy 3 chunk ĐẦU KHO."""
    lay = chunk_A[:3]
    cau_vang = next(c for c in VANG if c["cau_hoi"] == cau_hoi)
    kho = gop(" || ".join(lay))
    thay = [t for t in cau_vang["trich_dan"] if gop(t) in kho]
    if not cau_vang["trich_dan"] or not thay:
        return ("Không tìm thấy thông tin trong tài liệu.", lay)
    tu_khoa = [t for t in cau_vang["tu_khoa"] if gop(t).lower() in kho.lower()]
    return (f'{" ".join(tu_khoa)} — theo tài liệu [1].', lay)


def baseline_luon_tu_choi(cau_hoi):
    """Luôn nói 'không tìm thấy'. Nó ăn trọn 2 câu khong_co."""
    return ("Không tìm thấy thông tin trong tài liệu.", [])


bao_cao(chay_eval(baseline_lay_dau), "baseline: 3 chunk đầu")
bao_cao(chay_eval(baseline_luon_tu_choi), "baseline: luôn từ chối")
bao_cao(kq_A, "A: 300/0, vector")

> ⚠️ Nhìn dòng **"luôn từ chối"**. Một hệ thống **không làm gì cả** vẫn được
> điểm dương, chỉ nhờ trả lời đúng 2 câu `khong_co`.
>
> Đây chính là bẫy baseline của Phase 04 quay lại trong bộ áo RAG. Nếu bộ test
> của bạn có 40% câu không có đáp án, một hệ thống luôn từ chối sẽ đạt 0.40 —
> và ai đó sẽ đem con số đó đi khoe.
>
> **Luôn báo cáo kèm baseline.** "RAG đạt 0.85" là câu vô nghĩa cho tới khi biết
> baseline là bao nhiêu.

## 4. Sửa cấu hình theo đúng những gì đã học

Bài 1 nói chunk to hơn và có chồng lấn thì recall cao hơn. Bài 2 nói BM25 mạnh
với mã lỗi. Hãy áp dụng cả hai.

In [ ]:
def so_sanh_cau_hinh(truoc, sau):
    diem_sau = {r["id"]: r["diem"] for r in sau}
    chung = [r for r in truoc if r["id"] in diem_sau]
    ct, tl, gn = [], [], []
    for r in chung:
        a, b = r["diem"], diem_sau[r["id"]]
        (ct if b > a else tl if b < a else gn).append(r["id"])
    n = len(chung)
    return {"diem_truoc": sum(r["diem"] for r in chung) / n,
            "diem_sau": sum(diem_sau[r["id"]] for r in chung) / n,
            "cai_thien": ct, "thoai_lui": tl, "giu_nguyen": gn}


tim_C, chunk_C, nguon_C = dung_he_tim(600, 120, "bm25")
kq_C = chay_eval(tao_trinh_sinh(tim_C, chunk_C, nguon_C))

bao_cao(kq_A, "A: 300/0, vector")
bao_cao(kq_C, "C: 600/120, bm25")

ss1 = so_sanh_cau_hinh(kq_A, kq_C)
print()
print(f'✅ Cải thiện : {ss1["cai_thien"]}')
print(f'⚠️  THOÁI LUI : {ss1["thoai_lui"]}')

Điểm nhảy vọt. Nhưng **ngay cả một cải tiến rõ ràng như thế cũng có câu
thoái lui** — đổi sang BM25 làm hỏng đúng những câu mà embedding vốn mạnh.

Đây chưa phải điều đáng lo nhất. Hãy làm tiếp bước mà bài 2 đã khuyên: **hybrid**.

## 5. ⭐ Bước cải tiến làm hỏng thứ đang chạy tốt

Bài 2 đã đo: hybrid có recall trung bình cao nhất. Vậy chuyển từ BM25 sang
hybrid là một nâng cấp hiển nhiên.

In [ ]:
tim_B, chunk_B, nguon_B = dung_he_tim(600, 120, "hybrid")
kq_B = chay_eval(tao_trinh_sinh(tim_B, chunk_B, nguon_B))

bao_cao(kq_C, "C: 600/120, bm25")
bao_cao(kq_B, "B: 600/120, hybrid")

Điểm tổng tăng. Nếu dừng ở đây, bạn deploy hybrid và yên tâm.

**Đừng dừng ở đây.**

In [ ]:
ss = so_sanh_cau_hinh(kq_C, kq_B)
print(f'Điểm: {ss["diem_truoc"]:.2f}  ->  {ss["diem_sau"]:.2f}   (TĂNG)')
print(f'✅ Cải thiện : {ss["cai_thien"]}')
print(f'⚠️  THOÁI LUI : {ss["thoai_lui"]}')
print(f'   Giữ nguyên: {len(ss["giu_nguyen"])} câu')

In [ ]:
# Câu nào thoái lui, và nó hỏng ở KHÂU nào?
diem_C = {r["id"]: r for r in kq_C}
diem_B = {r["id"]: r for r in kq_B}

for ma_id in ss["thoai_lui"]:
    c = next(x for x in VANG if x["id"] == ma_id)
    print(f'[{ma_id}] ({c["loai"]}) {c["cau_hoi"]}')
    for ten, r in (("C (bm25) ", diem_C[ma_id]), ("B (hybrid)", diem_B[ma_id])):
        ct = r["chi_tiet"]
        print(f'   {ten}: điểm={r["diem"]:.1f}  recall={ct["recall"]:.2f}  '
              f'từ khoá={ct["du_tu_khoa"]}')
    print(f'   ghi chú bộ test: {c["ghi_chu"]}')
    print()

> ⭐ **Đây là bài học trung tâm của cả Phase 07 lẫn Phase 08:**
>
> **Điểm trung bình tăng KHÔNG có nghĩa là deploy được.** Ở đây trung bình tăng
> trong khi một câu mã lỗi đang chạy **hoàn hảo** bị hỏng — đúng như bài 2 đã
> cảnh báo: RRF kéo kết quả của bên mù tịt lên và đẩy đoạn đúng ra khỏi top-3.
>
> Chỉ có so sánh **theo từng câu** mới cho bạn thấy điều đó. Nhìn con số tổng
> 0.90 → 0.94, bạn sẽ deploy mà không hề biết.

**Và khi có câu thoái lui, việc tiếp theo KHÔNG phải là quay về cấu hình cũ.**
Việc tiếp theo là đọc từng câu hỏng rồi quyết định:

- Câu bị hỏng thuộc loại người dùng hay hỏi không? (Nhân viên hỗ trợ kỹ thuật
  hỏi mã lỗi suốt ngày — nên câu này **quan trọng hơn** trọng số 1/16 của nó.)
- Có cách nào lấy cả hai không? (Ví dụ: phát hiện mã định danh bằng regex rồi
  cho BM25 trọng số cao hơn với riêng những câu đó.)

> 📌 **Bộ eval không quyết định thay bạn.** Nó chỉ đảm bảo bạn quyết định trong
> lúc **biết** mình đang đánh đổi cái gì.

## 6. Hỏng ở retrieval hay ở sinh câu trả lời?

Bảng dưới tách hai loại lỗi ra. Với trình sinh giả lập (hoàn hảo), gần như mọi
lỗi phải nằm ở cột retrieval — nếu không thì bộ chấm của bạn có bug.

In [ ]:
def phan_tich_loi(kq, ten):
    loi_retrieval = loi_sinh = tot = 0
    for r in kq:
        ct = r["chi_tiet"]
        if not ct:
            continue
        if r["diem"] >= 0.99:
            tot += 1
        elif ct["recall"] < 1.0:
            loi_retrieval += 1
        else:
            loi_sinh += 1
    print(f"{ten:<24} đạt={tot:<3} lỗi retrieval={loi_retrieval:<3} lỗi sinh={loi_sinh}")


phan_tich_loi(kq_A, "A: 300/0, vector")
phan_tich_loi(kq_B, "B: 600/120, hybrid")

> 📌 Cột nào lớn thì đi sửa chỗ đó. Nghe hiển nhiên, nhưng phần lớn người mới
> gặp điểm thấp là lao vào sửa prompt — trong khi bảng này chỉ thẳng vào
> chunking và tìm kiếm.
>
> **Một con số duy nhất không bao giờ nói cho bạn biết đi sửa chỗ nào.**

## 7. Với API key: model thật hỏng thêm ở đâu?

Trình sinh giả lập là **trần trên**. Model thật sẽ thấp hơn — và khoảng cách đó
chính là phần việc của prompt engineering.

In [ ]:
import os

import anthropic

MODEL_RE = "claude-haiku-4-5"
CO_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
client = anthropic.Anthropic() if CO_KEY else None

HUONG_DAN = """Trả lời câu hỏi CHỈ dựa trên các đoạn được cung cấp.
Mỗi ý phải kèm số đoạn dạng [n]; chỉ dùng số đoạn CÓ THẬT.
Nếu các đoạn không đủ thông tin, trả lời đúng câu:
"Không tìm thấy thông tin trong tài liệu."
Trả lời ngắn gọn bằng tiếng Việt."""


def tra_loi_that(cau_hoi, k=3):
    idx = tim_B(cau_hoi, k)
    lay = [chunk_B[i] for i in idx]
    nc = "\n\n".join(f"[{i}] ({nguon_B[j]})\n{chunk_B[j]}"
                      for i, j in enumerate(idx, 1))
    r = client.messages.create(
        model=MODEL_RE, max_tokens=500, system=HUONG_DAN,
        messages=[{"role": "user",
                   "content": f"<tai_lieu>\n{nc}\n</tai_lieu>\n\nCâu hỏi: {cau_hoi}"}],
    )
    return "".join(b.text for b in r.content if b.type == "text"), lay


if CO_KEY:
    kq_that = chay_eval(tra_loi_that)
    bao_cao(kq_B, "B: trình sinh giả lập")
    bao_cao(kq_that, "B: model thật")
    print()
    ss2 = so_sanh_cau_hinh(kq_B, kq_that)
    print(f'Model thật kém hơn ở: {ss2["thoai_lui"]}')
else:
    print("⏭  Chưa có ANTHROPIC_API_KEY — bỏ qua phần chạy thật.")
    print("   Ước tính chi phí nếu chạy: 18 câu × ~700 token vào ≈ $0.02")
    print()
    print("   (minh hoạ) Kết quả thường thấy:")
    print("     B: trình sinh giả lập   TỔNG=0.94")
    print("     B: model thật           TỔNG=0.80 – 0.90")
    print("   Khoảng cách đó là phần model chưa nói đủ ý, quên trích dẫn,")
    print("   hoặc trả lời câu lẽ ra phải từ chối.")

## 8. Bịa số — cách kiểm rẻ mà hiệu quả bất ngờ

Phần lớn ảo giác trong tài liệu nội bộ là ảo giác về **con số**: "12 ngày phép"
thành "15 ngày phép". Và con số thì kiểm bằng code được.

In [ ]:
MAU_SO = re.compile(r"\d[\d.,]*")


def so_bia(cau_tra_loi, cac_chunk_lay):
    """Các con số xuất hiện trong câu trả lời mà KHÔNG có trong ngữ cảnh."""
    kho = gop(" || ".join(cac_chunk_lay))
    trong_kho = set(MAU_SO.findall(kho))
    trong_tl = set(MAU_SO.findall(cau_tra_loi))
    # Bỏ qua số trích dẫn [n] - chúng không phải dữ kiện
    so_trich_dan = set(MAU_TRICH_DAN.findall(cau_tra_loi))
    return sorted(trong_tl - trong_kho - so_trich_dan)


lay_thu = [chunk_B[i] for i in tim_B("Nhân viên có bao nhiêu ngày phép năm?", 3)]
for tl_thu in ["Phép năm là 12 ngày [1].",
               "Phép năm là 15 ngày, cộng dồn tối đa 9 ngày [1]."]:
    print(f"{tl_thu}\n   số không có trong ngữ cảnh: {so_bia(tl_thu, lay_thu)}")

> ⚠️ **Giới hạn thành thật của cách kiểm này:** nó chỉ bắt được số *không có
> trong ngữ cảnh*. Nếu model lấy nhầm một con số **có thật nhưng ở đoạn khác**
> (ví dụ trả lời "3 ngày" cho câu hỏi về phép năm, vì "3 ngày" có trong đoạn về
> nghỉ ốm) thì cách này **không bắt được**.
>
> Đó là lý do cần thêm **LLM-as-judge**: đưa (ngữ cảnh, câu trả lời) cho một
> model và hỏi *"mọi khẳng định có được đoạn văn chống lưng không?"*.
>
> Và **judge cũng phải được kiểm tra**: cho nó chấm vài ca bạn đã biết đáp án,
> gồm cả ca bạn cố tình làm sai. Một judge luôn nói "đạt" còn tệ hơn không có
> judge, vì nó cho bạn cảm giác an toàn giả.

## Tóm tắt

| Điều | Nhớ gì |
|---|---|
| **Tách retrieval khỏi sinh** | Một con số không cho bạn biết đi sửa chỗ nào |
| **Baseline** | "Luôn từ chối" vẫn được điểm dương. Không có baseline thì mọi con số vô nghĩa |
| **So sánh theo từng câu** | Trung bình tăng vẫn có thể đang giấu câu bị hỏng |
| **Trình sinh hoàn hảo** | Cô lập một biến — mọi điểm mất là lỗi retrieval |
| **Kiểm số bịa** | Rẻ, bắt được phần lớn ảo giác về dữ kiện |
| **LLM-as-judge** | Cần cho phần còn lại — nhưng phải kiểm chính judge trước |

---

## Xong Phase 08

**Bài tập:** `curriculum/08-rag/exercises/ex03_eval_rag.py` ⭐⭐

```powershell
pytest tests/phase08 -v
```

**Tiếp theo:**

- **Mini-project:** `curriculum/08-rag/mini_project/README.md`
- 🎯 **Project P6:** `projects/P6-rag-qa/README.md`

> Bài test khắt khe nhất cho những gì bạn vừa học:
>
> Đưa hệ RAG cho một đồng nghiệp dùng 15 phút. Ghi lại mọi câu nó trả lời sai,
> rồi **biến từng câu đó thành một case trong bộ câu hỏi vàng**. Bộ test của bạn
> sẽ tốt lên theo đúng cách mà sản phẩm thật tốt lên.